<a href="https://colab.research.google.com/github/skanrrking/DiabetesPrediction/blob/main/Logistic_Regression_Diabetes_Checkup_Predictor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

# ============================================
# PART 1: DIABETES RISK PREDICTION MODEL
# UCI CDC Diabetes Health Indicators (ID 891)
# ============================================

!pip install -q ucimlrepo

import pandas as pd
from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    accuracy_score,
    recall_score,
    precision_score
)

# 1. FETCH DATASET
dataset = fetch_ucirepo(id=891)

X = dataset.data.features.copy()
y = dataset.data.targets["Diabetes_binary"].astype(int)

print("Dataset loaded successfully!")
print("Features:", X.shape)
print("Target:", y.shape)
print("\nClass distribution:")
print(y.value_counts(normalize=True).sort_index().rename(
    index={0: "No diabetes (0)", 1: "Prediabetes or diabetes (1)"}
).map(lambda x: f"{x:.2%}"))

assert len(X) == 253680, f"Unexpected row count: {len(X)}"
assert X.shape[1] == 21, f"Expected 21 features, got {X.shape[1]}"

# 2. TRAIN/TEST SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(f"\nTraining samples: {len(X_train)}")
print(f"Testing samples: {len(X_test)}")

# 3. TRAIN RANDOM FOREST MODEL
diabetes_model = RandomForestClassifier(
    n_estimators=200,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

print("\nTraining model...")
diabetes_model.fit(X_train, y_train)
print("Training complete!")

# 4. EVALUATE RANDOM FOREST
y_pred = diabetes_model.predict(X_test)
y_prob = diabetes_model.predict_proba(X_test)[:, 1]

print("\n========== RANDOM FOREST ==========")
print(classification_report(
    y_test,
    y_pred,
    labels=[0, 1],
    target_names=["No diabetes (0)", "Prediabetes or diabetes (1)"],
    zero_division=0,
    digits=4
))

model_accuracy = accuracy_score(y_test, y_pred)
model_recall = recall_score(y_test, y_pred, pos_label=1)
model_precision = precision_score(
    y_test, y_pred, pos_label=1, zero_division=0
)
model_auc = roc_auc_score(y_test, y_prob)

print(f"Accuracy:  {model_accuracy:.4f}")
print(f"Recall (Class 1): {model_recall:.4f}")
print(f"Precision (Class 1): {model_precision:.4f}")
print(f"ROC-AUC: {model_auc:.4f}")

# 5. BASELINE: ALWAYS PREDICT NO DIABETES
baseline_pred = [0] * len(y_test)

baseline_accuracy = accuracy_score(y_test, baseline_pred)
baseline_recall = recall_score(
    y_test, baseline_pred, pos_label=1, zero_division=0
)
baseline_precision = precision_score(
    y_test, baseline_pred, pos_label=1, zero_division=0
)

print("\n========== BASELINE MODEL ==========")
print("Prediction: Always no diabetes (class 0)")
print(f"Accuracy:  {baseline_accuracy:.4f} ({baseline_accuracy:.2%})")
print(f"Recall (Class 1): {baseline_recall:.4f}")
print(f"Precision (Class 1): {baseline_precision:.4f}")

# 6. SIDE-BY-SIDE COMPARISON
comparison = pd.DataFrame({
    "Metric": ["Accuracy", "Recall (Class 1)", "Precision (Class 1)"],
    "Always Predict No Diabetes": [
        baseline_accuracy, baseline_recall, baseline_precision
    ],
    "Random Forest": [
        model_accuracy, model_recall, model_precision
    ]
})

print("\n========== MODEL COMPARISON ==========")
print(comparison.to_string(index=False, formatters={
    "Always Predict No Diabetes": "{:.4f}".format,
    "Random Forest": "{:.4f}".format
}))

print("\nModel saved in variable: diabetes_model")


In [ ]:

# ============================================
# PART 2: COMPARE THREE MACHINE LEARNING MODELS
# Random Forest vs. Logistic Regression vs. XGBoost
# ============================================

!pip install -q xgboost

import pandas as pd

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)
from xgboost import XGBClassifier

# 1. Calculate class imbalance from training data
negative_count = (y_train == 0).sum()
positive_count = (y_train == 1).sum()

scale_pos_weight = negative_count / positive_count

print(f"Training class 0: {negative_count:,}")
print(f"Training class 1: {positive_count:,}")
print(f"XGBoost class weight ratio: {scale_pos_weight:.3f}")

# 2. Create Logistic Regression model
# StandardScaler standardizes features for Logistic Regression.
logistic_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(
        class_weight="balanced",
        max_iter=2000,
        random_state=42
    )
)

# 3. Create XGBoost model
xgboost_model = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

# 4. Train the two new models
print("\nTraining Logistic Regression...")
logistic_model.fit(X_train, y_train)

print("Training XGBoost...")
xgboost_model.fit(X_train, y_train)

print("Training complete!")

# 5. Compare all three trained models
models = {
    "Random Forest": diabetes_model,
    "Logistic Regression": logistic_model,
    "XGBoost": xgboost_model
}

results = []

for name, model in models.items():
    predictions = model.predict(X_test)
    probabilities = model.predict_proba(X_test)[:, 1]

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision (Class 1)": precision_score(
            y_test, predictions, pos_label=1, zero_division=0
        ),
        "Recall (Class 1)": recall_score(
            y_test, predictions, pos_label=1, zero_division=0
        ),
        "F1-score (Class 1)": f1_score(
            y_test, predictions, pos_label=1, zero_division=0
        ),
        "ROC-AUC": roc_auc_score(y_test, probabilities)
    })

# 6. Add baseline: always predict no diabetes (class 0)
baseline_predictions = [0] * len(y_test)

results.append({
    "Model": "Baseline (always class 0)",
    "Accuracy": accuracy_score(y_test, baseline_predictions),
    "Precision (Class 1)": precision_score(
        y_test, baseline_predictions, pos_label=1, zero_division=0
    ),
    "Recall (Class 1)": recall_score(
        y_test, baseline_predictions, pos_label=1, zero_division=0
    ),
    "F1-score (Class 1)": f1_score(
        y_test, baseline_predictions, pos_label=1, zero_division=0
    ),
    # Constant predictions cannot rank positive vs. negative cases.
    "ROC-AUC": float("nan")
})

# 7. Display comparison, highest class 1 recall first
results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="Recall (Class 1)",
    ascending=False
).reset_index(drop=True)

print("\n========== MODEL COMPARISON ==========")
print(results_df.to_string(
    index=False,
    formatters={
        col: "{:.4f}".format
        for col in results_df.columns
        if col != "Model"
    }
))

# 8. Identify the best model for each key metric
print("\n========== BEST MODEL BY METRIC ==========")

for metric in [
    "Accuracy",
    "Precision (Class 1)",
    "Recall (Class 1)",
    "F1-score (Class 1)",
    "ROC-AUC"
]:
    valid_results = results_df.dropna(subset=[metric])
    best_row = valid_results.loc[valid_results[metric].idxmax()]
    print(f"{metric}: {best_row['Model']} ({best_row[metric]:.4f})")

print("\nModels available for future cells:")
print("- diabetes_model (Random Forest)")
print("- logistic_model (Logistic Regression)")
print("- xgboost_model (XGBoost)")


In [ ]:

# ============================================
# PART 3: CONFUSION MATRICES + PRECISION/RECALL
# ============================================

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
    precision_score,
    recall_score
)

# The three trained models from Part 1 and Part 2
models = {
    "Random Forest": diabetes_model,
    "Logistic Regression": logistic_model,
    "XGBoost": xgboost_model
}

# 1. Display confusion matrices
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, model) in zip(axes, models.items()):
    predictions = model.predict(X_test)

    cm = confusion_matrix(
        y_test,
        predictions,
        labels=[0, 1]
    )

    display = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=["No Diabetes (0)", "Positive (1)"]
    )

    display.plot(ax=ax, cmap="Blues", values_format="d", colorbar=False)
    ax.set_title(name)
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("Actual label")
    ax.tick_params(axis="x", rotation=15)

plt.suptitle("Confusion Matrix Comparison", fontsize=16)
plt.tight_layout()
plt.show()

# 2. Compare precision and recall for class 1
comparison = []

for name, model in models.items():
    predictions = model.predict(X_test)

    precision = precision_score(
        y_test, predictions, pos_label=1, zero_division=0
    )
    recall = recall_score(
        y_test, predictions, pos_label=1, zero_division=0
    )

    comparison.append({
        "Model": name,
        "Precision (Class 1)": precision,
        "Recall (Class 1)": recall
    })

comparison_df = pd.DataFrame(comparison)

print("\n========== PRECISION & RECALL ==========")
print(comparison_df.to_string(
    index=False,
    formatters={
        "Precision (Class 1)": "{:.4f}".format,
        "Recall (Class 1)": "{:.4f}".format
    }
))

# 3. Bar chart comparing precision and recall
comparison_df.set_index("Model").plot(
    kind="bar",
    figsize=(10, 5),
    ylim=(0, 1),
    rot=0
)

plt.title("Precision vs. Recall for Class 1")
plt.ylabel("Score")
plt.xlabel("Model")
plt.legend(loc="best")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:

import pandas as pd
import matplotlib.pyplot as plt

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    recall_score,
    precision_score,
    confusion_matrix,
    ConfusionMatrixDisplay
)

# Tier 1: self-reported health and lifestyle questions
tier1_features = [
    "BMI",
    "Smoker",
    "PhysActivity",
    "Fruits",
    "Veggies",
    "HvyAlcoholConsump",
    "GenHlth",
    "MentHlth",
    "PhysHlth",
    "DiffWalk",
    "Age",
    "Sex"
]

# Tier 2: medical history and clinical information
tier2_features = [
    "HighBP",
    "HighChol",
    "CholCheck",
    "Stroke",
    "HeartDiseaseorAttack"
]

feature_sets = {
    "Tier 1 only": tier1_features,
    "Tier 1 + Tier 2": tier1_features + tier2_features
}

results = []
predictions_by_model = {}

for name, features in feature_sets.items():

    # Create the Logistic Regression model
    model = make_pipeline(
        StandardScaler(),
        LogisticRegression(
            class_weight="balanced",
            max_iter=2000,
            random_state=42
        )
    )

    # Train using only the selected questions
    model.fit(X_train[features], y_train)

    # Predict on the test data
    predictions = model.predict(X_test[features])

    # Calculate the confusion matrix
    cm = confusion_matrix(y_test, predictions, labels=[0, 1])

    # Save predictions for the confusion matrix plots
    predictions_by_model[name] = predictions

    # Save evaluation metrics
    results.append({
        "Model": name,
        "Number of Questions": len(features),
        "Recall": recall_score(y_test, predictions, zero_division=0),
        "Precision": precision_score(
            y_test, predictions, zero_division=0
        ),
        "TN": cm[0, 0],
        "FP": cm[0, 1],
        "FN": cm[1, 0],
        "TP": cm[1, 1]
    })

# Display results
results_df = pd.DataFrame(results)

print("MODEL COMPARISON")
print(results_df.round(4).to_string(index=False))

# Plot both confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, (name, features) in zip(axes, feature_sets.items()):
    cm = confusion_matrix(
        y_test,
        predictions_by_model[name],
        labels=[0, 1]
    )

    display = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=["No diabetes", "Prediabetes/diabetes"]
    )

    display.plot(ax=ax, values_format="d", colorbar=False)
    ax.set_title(f"{name} ({len(features)} questions)")

plt.tight_layout()
plt.show()


In [ ]:

import pandas as pd

# Get the Logistic Regression model from the pipeline
lr = logistic_model.named_steps["logisticregression"]

# Use all features from the original model
importance_df = pd.DataFrame({
    "Feature": X_train.columns,
    "Coefficient": lr.coef_[0],
    "Absolute Importance": abs(lr.coef_[0])
})

# Rank features from most to least important
importance_df = importance_df.sort_values(
    by="Absolute Importance",
    ascending=False
)

print(importance_df.round(4).to_string(index=False))


In [ ]:

import pandas as pd
import matplotlib.pyplot as plt

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    precision_score,
    recall_score,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay
)

# Features ranked by importance from your original model
ranked_features = [
    "GenHlth",
    "BMI",
    "Age",
    "HighBP",
    "HighChol",
    "CholCheck",
    "HvyAlcoholConsump",
    "Sex",
    "Income"
]

# Create the four feature sets
feature_sets = {
    "Top 3 Features": ranked_features[:3],
    "Top 5 Features": ranked_features[:5],
    "Top 7 Features": ranked_features[:7],
    "Top 9 Features": ranked_features[:9]
}

results = []
predictions_by_model = {}

# Train and evaluate each model
for name, features in feature_sets.items():

    model = make_pipeline(
        StandardScaler(),
        LogisticRegression(
            class_weight="balanced",
            max_iter=2000,
            random_state=42
        )
    )

    # Train the model
    model.fit(X_train[features], y_train)

    # Predict class labels
    predictions = model.predict(X_test[features])

    # Predict probabilities for ROC-AUC
    probabilities = model.predict_proba(X_test[features])[:, 1]

    # Calculate evaluation metrics
    precision = precision_score(
        y_test, predictions, zero_division=0
    )
    recall = recall_score(
        y_test, predictions, zero_division=0
    )
    roc_auc = roc_auc_score(y_test, probabilities)

    # Calculate confusion matrix
    cm = confusion_matrix(y_test, predictions, labels=[0, 1])

    predictions_by_model[name] = predictions

    results.append({
        "Model": name,
        "Number of Features": len(features),
        "Precision": precision,
        "Recall": recall,
        "ROC-AUC": roc_auc,
        "TN": cm[0, 0],
        "FP": cm[0, 1],
        "FN": cm[1, 0],
        "TP": cm[1, 1]
    })

# Display comparison table
results_df = pd.DataFrame(results)

print("MODEL COMPARISON")
print(results_df.round(4).to_string(index=False))

# Display all four confusion matrices
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for ax, (name, features) in zip(axes, feature_sets.items()):

    cm = confusion_matrix(
        y_test,
        predictions_by_model[name],
        labels=[0, 1]
    )

    display = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=["No diabetes", "Prediabetes/diabetes"]
    )

    display.plot(
        ax=ax,
        values_format="d",
        colorbar=False
    )

    ax.set_title(f"{name} ({len(features)} features)")

plt.tight_layout()
plt.show()


In [ ]:

import matplotlib.pyplot as plt
from sklearn.metrics import (
    precision_recall_curve,
    average_precision_score
)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Define the two feature sets
top5_features = [
    "GenHlth",
    "BMI",
    "Age",
    "HighBP",
    "HighChol"
]

all_features = list(X_train.columns)

# Train the top 5 model
top5_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(
        class_weight="balanced",
        max_iter=2000,
        random_state=42
    )
)
top5_model.fit(X_train[top5_features], y_train)

# Train the full 21-feature model
full_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(
        class_weight="balanced",
        max_iter=2000,
        random_state=42
    )
)
full_model.fit(X_train[all_features], y_train)

# Get predicted probabilities for the positive class
top5_probs = top5_model.predict_proba(X_test[top5_features])[:, 1]
full_probs = full_model.predict_proba(X_test[all_features])[:, 1]

# Calculate precision-recall curves
top5_precision, top5_recall, _ = precision_recall_curve(
    y_test, top5_probs
)
full_precision, full_recall, _ = precision_recall_curve(
    y_test, full_probs
)

# Calculate Average Precision (AP)
top5_ap = average_precision_score(y_test, top5_probs)
full_ap = average_precision_score(y_test, full_probs)

# Plot both curves
plt.figure(figsize=(9, 6))

plt.plot(
    top5_recall,
    top5_precision,
    label=f"Top 5 Features (AP = {top5_ap:.3f})"
)

plt.plot(
    full_recall,
    full_precision,
    label=f"Full 21 Features (AP = {full_ap:.3f})"
)

# Baseline: positive-class proportion
baseline = y_test.mean()
plt.axhline(
    y=baseline,
    linestyle="--",
    label=f"Baseline (Prevalence = {baseline:.3f})"
)

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision–Recall Curve: Top 5 vs. Full 21 Features")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Top 5 Features Average Precision: {top5_ap:.4f}")
print(f"Full 21 Features Average Precision: {full_ap:.4f}")


In [ ]:

import pandas as pd
import matplotlib.pyplot as plt

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import recall_score

# Top 5 features
top5_features = [
    "GenHlth",
    "BMI",
    "Age",
    "HighBP",
    "HighChol"
]

# Train the top 5 Logistic Regression model
top5_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(
        class_weight="balanced",
        max_iter=2000,
        random_state=42
    )
)

top5_model.fit(X_train[top5_features], y_train)

# Predict on the test set
predictions = top5_model.predict(X_test[top5_features])

# Combine test data, actual outcomes, and predictions
evaluation_df = X_test.copy().reset_index(drop=True)
evaluation_df["Actual"] = pd.Series(y_test).reset_index(drop=True)
evaluation_df["Predicted"] = predictions

# Convert encoded age categories into age bands
age_labels = {
    1: "18-24", 2: "25-29", 3: "30-34",
    4: "35-39", 5: "40-44", 6: "45-49",
    7: "50-54", 8: "55-59", 9: "60-64",
    10: "65-69", 11: "70-74",
    12: "75-79", 13: "80+"
}

evaluation_df["Age Category"] = evaluation_df["Age"].map(age_labels)

# Combine age categories into four requested groups
age_group_map = {
    "18-24": "18-39",
    "25-29": "18-39",
    "30-34": "18-39",
    "35-39": "18-39",
    "40-44": "40-54",
    "45-49": "40-54",
    "50-54": "40-54",
    "55-59": "55-69",
    "60-64": "55-69",
    "65-69": "55-69",
    "70-74": "70+",
    "75-79": "70+",
    "80+": "70+"
}

evaluation_df["Age Band"] = evaluation_df["Age Category"].map(age_group_map)

# Label sex categories
evaluation_df["Sex Group"] = evaluation_df["Sex"].map({
    0: "Female",
    1: "Male"
})

# Group income categories
evaluation_df["Income Group"] = pd.cut(
    evaluation_df["Income"],
    bins=[0, 3, 6, 8],
    labels=["Lower (1-3)", "Middle (4-6)", "Higher (7-8)"]
)

# Calculate recall for each group
def calculate_group_recall(data, group_column):
    rows = []

    for group, subset in data.groupby(group_column, observed=True):
        positive_count = int((subset["Actual"] == 1).sum())

        recall = (
            recall_score(subset["Actual"], subset["Predicted"])
            if positive_count > 0
            else float("nan")
        )

        rows.append({
            "Group": group,
            "Positive Cases": positive_count,
            "Detected": int(
                ((subset["Actual"] == 1) &
                 (subset["Predicted"] == 1)).sum()
            ),
            "Recall": recall
        })

    return pd.DataFrame(rows)

# Calculate and display results
groupings = {
    "Age Band": "Age Band",
    "Sex": "Sex Group",
    "Income Group": "Income Group"
}

results = {}

for title, column in groupings.items():
    result = calculate_group_recall(evaluation_df, column)
    results[title] = result

    print(f"\nRECALL BY {title.upper()}")
    print(result.round(3).to_string(index=False))

# Plot all three comparisons
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

for ax, (title, result) in zip(axes, results.items()):
    ax.bar(result["Group"].astype(str), result["Recall"])
    ax.set_title(f"Recall by {title}")
    ax.set_ylabel("Recall")
    ax.set_ylim(0, 1)
    ax.tick_params(axis="x", rotation=45)
    ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:

import pandas as pd
from sklearn.metrics import recall_score, precision_score

# Get predicted probabilities from the trained top 5 model
probabilities = top5_model.predict_proba(X_test[top5_features])[:, 1]

# Prepare evaluation data
evaluation_df = X_test.copy().reset_index(drop=True)
evaluation_df["Actual"] = pd.Series(y_test).reset_index(drop=True)
evaluation_df["Probability"] = probabilities

# Convert encoded age categories into age groups
age_labels = {
    1: "18-24", 2: "25-29", 3: "30-34",
    4: "35-39", 5: "40-44", 6: "45-49",
    7: "50-54", 8: "55-59", 9: "60-64",
    10: "65-69", 11: "70-74",
    12: "75-79", 13: "80+"
}

age_group_map = {
    "18-24": "18-39",
    "25-29": "18-39",
    "30-34": "18-39",
    "35-39": "18-39",
    "40-44": "40-54",
    "45-49": "40-54",
    "50-54": "40-54",
    "55-59": "55-69",
    "60-64": "55-69",
    "65-69": "55-69",
    "70-74": "70+",
    "75-79": "70+",
    "80+": "70+"
}

evaluation_df["Age Band"] = (
    evaluation_df["Age"].map(age_labels).map(age_group_map)
)

# Keep only people under 40
under_40 = evaluation_df[evaluation_df["Age Band"] == "18-39"].copy()

# Test thresholds from 0.5 down to 0.1
thresholds = [0.50, 0.45, 0.40, 0.35, 0.30, 0.25, 0.20, 0.15, 0.10]

results = []

for threshold in thresholds:
    predicted = (
        under_40["Probability"] >= threshold
    ).astype(int)

    actual = under_40["Actual"]

    results.append({
        "Threshold": threshold,
        "Positive Cases": int((actual == 1).sum()),
        "Detected": int(((actual == 1) & (predicted == 1)).sum()),
        "Recall": recall_score(actual, predicted, zero_division=0),
        "Precision": precision_score(actual, predicted, zero_division=0),
        "False Positives": int(((actual == 0) & (predicted == 1)).sum())
    })

results_df = pd.DataFrame(results)

print("RESULTS FOR AGES 18–39 ONLY")
print(results_df.round(3).to_string(index=False))


In [ ]:
# Get predicted probabilities
probabilities = top5_model.predict_proba(X_test[top5_features])[:, 1]

# Use threshold 0.25 for ages 18–39
# Use threshold 0.50 for ages 40+
age_thresholds = X_test["Age"].map(
    lambda age: 0.25 if age <= 4 else 0.50
)

# Generate predictions using age-specific thresholds
predictions = (probabilities >= age_thresholds).astype(int)

print("Predictions generated using age-specific thresholds.")

In [ ]:

%pip -q install gradio

import gradio as gr
import pandas as pd
import matplotlib.pyplot as plt

top5_features = ["GenHlth", "BMI", "Age", "HighBP", "HighChol"]

def age_to_category(years):
    if years <= 24: return 1
    if years <= 29: return 2
    if years <= 34: return 3
    if years <= 39: return 4
    if years <= 44: return 5
    if years <= 49: return 6
    if years <= 54: return 7
    if years <= 59: return 8
    if years <= 64: return 9
    if years <= 69: return 10
    if years <= 74: return 11
    if years <= 79: return 12
    return 13

def predict_diabetes(health, bp, cholesterol,
                     weight, weight_unit, height, height_unit, age):

    if any(v is None for v in [health, bp, cholesterol, weight, height, age]):
        return "Please answer every question.", None

    if weight <= 0 or height <= 0 or age < 18 or age > 120:
        return "Enter a positive height and weight and an age from 18 to 120.", None

    # Convert measurements
    weight_kg = weight * 0.45359237 if weight_unit == "lb" else weight
    height_m = height * 0.0254 if height_unit == "inches" else height / 100
    bmi = weight_kg / (height_m ** 2)
    age_code = age_to_category(int(age))

    person = pd.DataFrame([{
        "GenHlth": int(health),
        "BMI": bmi,
        "Age": age_code,
        "HighBP": int(bp),
        "HighChol": int(cholesterol)
    }], columns=top5_features)

    probability = top5_model.predict_proba(person)[0, 1]
    threshold = 0.25 if age_code <= 4 else 0.50
    prediction = probability >= threshold

    if prediction:
        message = (
            "**Suggest a diabetes check-up**\n\n"
            "*This project is a prototype and is not medical advice.*"
        )
    else:
        message = (
            "**No check-up suggested**\n"
            "This does not rule out diabetes or mean screening is unnecessary.\n\n"
            "*This project is a prototype and is not medical advice.*"
        )

    message += (
        f"\n\n**Risk Score:** {probability:.1%}"
        f" (**Body mass index:** {bmi:.1f})"
    )

    # Feature contributions to model log-odds
    scaler = top5_model.named_steps["standardscaler"]
    classifier = top5_model.named_steps["logisticregression"]
    standardized = scaler.transform(person)[0]
    contributions = standardized * classifier.coef_[0]

    labels = [
        "General health", "BMI", "Age",
        "High blood pressure", "High cholesterol"
    ]

    chart_df = pd.DataFrame({
        "Feature": labels,
        "Contribution": contributions
    }).sort_values("Contribution")

    # Red = pushes toward positive class; green = pushes away
    colors = [
        "red" if value > 0 else "green"
        for value in chart_df["Contribution"]
    ]

    fig, ax = plt.subplots(figsize=(6, 3))
    ax.barh(chart_df["Feature"], chart_df["Contribution"], color=colors)
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_xlabel("Lowers risk score     |     Raises risk score")
    ax.set_title("How Your Answers Move the Risk Score")
    ax.grid(axis="x", alpha=0.25)
    fig.tight_layout()

    return message, fig


with gr.Blocks(title="Diabetes Risk Education Tool") as demo:
    gr.Markdown("# Diabetes Risk Education Tool")
    gr.Markdown(
        "Answer the questions below. BMI is calculated automatically. "
        "This tool provides an educational model estimate, not a diagnosis."
    )

    health = gr.Radio(
        choices=[
            ("Excellent", 1), ("Very good", 2), ("Good", 3),
            ("Fair", 4), ("Poor", 5)
        ],
        value=3,
        label="1. How would you describe your general health?"
    )

    with gr.Row():
        bp = gr.Radio(
            choices=[("No", 0), ("Yes", 1)],
            value=0,
            label="2. Do you have high blood pressure?"
        )
        cholesterol = gr.Radio(
            choices=[("No", 0), ("Yes", 1)],
            value=0,
            label="3. Do you have high cholesterol?"
        )

    with gr.Row():
        weight = gr.Number(value=150, label="4. Weight")
        weight_unit = gr.Radio(
            choices=["lb", "kg"], value="lb", label="Weight unit"
        )

    with gr.Row():
        height = gr.Number(value=65, label="5. Height")
        height_unit = gr.Radio(
            choices=["inches", "cm"], value="inches", label="Height unit"
        )

    age = gr.Number(
        value=50,
        precision=0,
        label="6. Age (years)"
    )

    submit = gr.Button("Evaluate Responses", variant="primary")
    result = gr.Markdown()
    chart = gr.Plot(label="Feature Contributions")

    submit.click(
        fn=predict_diabetes,
        inputs=[
            health, bp, cholesterol, weight, weight_unit,
            height, height_unit, age
        ],
        outputs=[result, chart]
    )

demo.launch(share=True)
